# Camada Gold – Modelagem Analítica e Consolidação de Negócio


In [0]:
from pyspark.sql import functions as F, Window

# Leitura das tabelas Silver
dim_consumidores_df  = spark.table("workspace.silver.dim_consumidores")
fat_pedidos_df       = spark.table("workspace.silver.fat_pedidos")
fat_itens_df         = spark.table("workspace.silver.fat_itens_pedidos")
fat_pagamentos_df    = spark.table("workspace.silver.fat_pagamentos_pedidos")
fat_avaliacoes_df    = spark.table("workspace.silver.fat_avaliacoes_pedidos")
dim_produtos_df      = spark.table("workspace.silver.dim_produtos")
dim_vendedores_df    = spark.table("workspace.silver.dim_vendedores")
dim_traducao_df      = spark.table("workspace.silver.dim_categoria_produtos_traducao")
dim_cotacao_df       = spark.table("workspace.silver.dim_cotacao_dolar")
fat_pedido_total_df  = spark.table("workspace.silver.fat_pedido_total")

# Catalog e schema Gold
catalogo     = "workspace"
gold_db_name = "gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalogo}.{gold_db_name}")
print("Tabelas Silver carregadas com sucesso!\n")

---
## 1º Projeto: Visão Comercial e Volume de Produtos

### `gold.fat_vendas_comercial` — Tabela Principal
Granularidade: Ano, Mês e Categoria de Produto.

In [0]:
# Enriquece itens com categoria do produto
itens_enrich_df = (
    fat_itens_df
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
)

# Enriquece com data e id_consumidor via pedidos
itens_pedidos_df = (
    itens_enrich_df
    .join(
        fat_pedidos_df.select("id_pedido", "id_consumidor", "data_compra"),
        on="id_pedido", how="inner"
    )
    .withColumn("ano_venda", F.year("data_compra"))
    .withColumn("mes_venda", F.month("data_compra"))
)

# Enriquece com cotação do dólar
itens_completo_df = (
    itens_pedidos_df
    .join(
        dim_cotacao_df.select("data_cotacao", "cotacao_compra"),
        F.to_date("data_compra") == F.col("data_cotacao"),
        how="left"
    )
    .withColumn("receita_item_usd", F.col("preco_BRL") / F.col("cotacao_compra"))
)

# Agrega na granularidade: Ano, Mês e Categoria
fat_vendas_df = (
    itens_completo_df
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
        F.round(F.sum("receita_item_usd"), 2).alias("receita_total_usd"),
        F.round(F.avg("preco_BRL"), 2).alias("ticket_medio_brl"),
    )
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
    .withColumn("data_ingestao", F.current_timestamp())
)

fat_vendas_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db_name}.fat_vendas_comercial")
print(f"✅ Tabela gold.fat_vendas_comercial criada com sucesso! ({fat_vendas_df.count()} registros)\n")

---
### Entrega 2 — Rankings Comerciais
Top 5 produtos mais e menos vendidos.

In [0]:
df_mais_vendidos = (
    fat_itens_df
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .groupBy("id_produto", "categoria_produto")
    .agg(F.count("id_item").alias("quantidade_vendida"))
    .orderBy(F.col("quantidade_vendida").desc())
    .limit(5)
)

print("🏆 TOP 5 PRODUTOS MAIS VENDIDOS")
display(df_mais_vendidos)

In [0]:
df_menos_vendidos = (
    fat_itens_df
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .groupBy("id_produto", "categoria_produto")
    .agg(F.count("id_item").alias("quantidade_vendida"))
    .orderBy(F.col("quantidade_vendida").asc())
    .limit(5)
)

print("📉 TOP 5 PRODUTOS MENOS VENDIDOS")
display(df_menos_vendidos)

---
## 2º Projeto: Satisfação de Clientes e Qualidade de Parceiros

### `gold.fat_avaliacoes_clientes` — Tabela Principal
Granularidade: Categoria do Produto, Nome do Vendedor e Estado do Vendedor.

In [0]:
# Enriquece avaliações com categoria e vendedor
aval_base_df = (
    fat_avaliacoes_df
    .join(fat_pedidos_df.select("id_pedido", "data_compra"), on="id_pedido", how="left")
    .join(fat_itens_df.select("id_pedido", "id_produto", "id_vendedor"), on="id_pedido", how="left")
    .join(dim_produtos_df.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .join(dim_vendedores_df.select("id_vendedor", "estado"), on="id_vendedor", how="left")
    .withColumnRenamed("estado", "estado_vendedor")
)

# Agrega na granularidade: Categoria, Vendedor e Estado
fat_aval_clientes_df = (
    aval_base_df
    .groupBy("categoria_produto", "id_vendedor", "estado_vendedor")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.count(F.when(F.col("nota_avaliacao") >= 4, True)).alias("total_avaliacoes_positivas"),
        F.count(F.when(F.col("nota_avaliacao") <= 2, True)).alias("total_avaliacoes_negativas"),
        F.round(
            F.count(F.when(F.col("nota_avaliacao") >= 4, True)) * 100.0 / F.count("id_avaliacao"), 2
        ).alias("percentual_satisfacao"),
    )
    .orderBy("categoria_produto")
    .withColumn("data_ingestao", F.current_timestamp())
)

fat_aval_clientes_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db_name}.fat_avaliacoes_clientes")
print(f"✅ Tabela gold.fat_avaliacoes_clientes criada com sucesso! ({fat_aval_clientes_df.count()} registros)\n")

---
### Entrega 2 — Rankings de Qualidade
Critério: nota média (DESC) + volume de avaliações como desempate (DESC).

In [0]:
rank_produto_df = (
    aval_base_df
    .groupBy("id_produto", "categoria_produto")
    .agg(
        F.round(F.avg("nota_avaliacao"), 4).alias("avaliacao_media"),
        F.count("id_avaliacao").alias("total_avaliacoes"),
    )
)

print("⭐ PRODUTO MAIS BEM AVALIADO")
display(
    rank_produto_df
    .orderBy(F.col("avaliacao_media").desc(), F.col("total_avaliacoes").desc())
    .limit(1)
)

print("💔 PRODUTO MENOS BEM AVALIADO")
display(
    rank_produto_df
    .orderBy(F.col("avaliacao_media").asc(), F.col("total_avaliacoes").desc())
    .limit(1)
)

In [0]:
rank_vendedor_df = (
    aval_base_df
    .groupBy("id_vendedor", "estado_vendedor")
    .agg(
        F.round(F.avg("nota_avaliacao"), 4).alias("avaliacao_media"),
        F.count("id_avaliacao").alias("total_avaliacoes"),
    )
)

print("🏅 VENDEDOR MAIS BEM AVALIADO")
display(
    rank_vendedor_df
    .orderBy(F.col("avaliacao_media").desc(), F.col("total_avaliacoes").desc())
    .limit(1)
)

print("👎 VENDEDOR MENOS BEM AVALIADO")
display(
    rank_vendedor_df
    .orderBy(F.col("avaliacao_media").asc(), F.col("total_avaliacoes").desc())
    .limit(1)
)

---
## Otimização das Tabelas Gold

In [0]:
tabelas_gold = [
    (f"{catalogo}.{gold_db_name}.fat_vendas_comercial",    "ano_venda, mes_venda"),
    (f"{catalogo}.{gold_db_name}.fat_avaliacoes_clientes", "categoria_produto"),
]

for tabela, colunas in tabelas_gold:
    print(f"Otimizando {tabela}...")
    spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({colunas})")
    print(f"  ✅ {tabela} otimizada.")

## Verificação Final

In [0]:
print("=" * 60)
print("TABELAS CRIADAS NA CAMADA GOLD")
print("=" * 60)
spark.sql(f"SHOW TABLES IN {catalogo}.{gold_db_name}").show(truncate=False)

print("\nProcesso da camada Gold concluído com sucesso!")